In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, create_index

documents = load_faq_data()
index = create_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    intructions=instructions,
)

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search('how do i run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after course started enrollment"}', call_id='call_pEcN8S5JwSTaiyUfFAZQtKnG', name='search', type='function_call', id='fc_01518ebbfc2d44e9006a30b842e5648190990ed143c2d95912', namespace=None, status='completed')]

In [9]:
call = response.output[0]

In [10]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after course started enrollment"}', call_id='call_pEcN8S5JwSTaiyUfFAZQtKnG', name='search', type='function_call', id='fc_01518ebbfc2d44e9006a30b842e5648190990ed143c2d95912', namespace=None, status='completed')

In [11]:
import json

args = json.loads(call.arguments)

In [29]:
args

{'query': 'join course discovered late can I join after course started enrollment'}

In [12]:
result = search(**args)

In [13]:
result_json = json.dumps(result)

In [30]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search analye the result then perform more searches if needed.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_Cjt0EH8sovZrSu2NLAfhD3zg', name='search', type='function_call', id='fc_04100cfebd6fe50e006a30b845e9d08196af7aa70134ab9541', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': '

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after course started enrollment"}', call_id='call_pEcN8S5JwSTaiyUfFAZQtKnG', name='search', type='function_call', id='fc_01518ebbfc2d44e9006a30b842e5648190990ed143c2d95912', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pEcN8S5JwSTaiyUfFAZQtKnG',
  'output': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "69d122f12e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?", "answer": "No, you can only get a certificat

In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)



In [17]:
response.output

[ResponseOutputMessage(id='msg_01518ebbfc2d44e9006a30b844a69c8190a1152add175ecc4c', content=[ResponseOutputText(annotations=[], text='Yes, you can still join. You can start learning and submitting homework while the form is open.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [18]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [19]:
messages.extend(response.output)

for item in response.output:
    if item.type=='function_call':
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type=='message':
        print('assiatnt')
        print(item.content[0].text)

assiatnt
Yes, you can still join. You can start learning and submitting homework while the form is open.

If you want a certificate, make sure you submit your project while submissions are still being accepted.


In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after course started enrollment"}', call_id='call_pEcN8S5JwSTaiyUfFAZQtKnG', name='search', type='function_call', id='fc_01518ebbfc2d44e9006a30b842e5648190990ed143c2d95912', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pEcN8S5JwSTaiyUfFAZQtKnG',
  'output': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "69d122f12e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?", "answer": "No, you can only get a certificat

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search analye the result then perform more searches if needed.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [22]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}


In [23]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just learning, you can also follow along even after discovering the course.

Would you like help with anything else about the course, like certificates, registration, or homework?


In [24]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. If you’re just learning, you can follow the course even if you discovered it late.

Would you like to explore anything else about the course, such as certificates, homework, or the next cohort?


In [25]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [26]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 ollama run llama3 local server Python client"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` installer from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` installer from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you can restart the server with:\n```bash\nollama serve\n```\n\nIf you want, I can also show you how to us

In [27]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join the course late discovered course can I still join enrollment late start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. If you’re joining later, it’s still worth participating even if you miss some earlier parts.

If you’d like, I can also help explain the certificate requirements or how to catch up quickly. Is there anything else you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted. If you’re joining later, it’s still worth participating even if you miss some earlier parts.\n\nIf you’d like, I can also help explain the certificate requirements or how to catch up quickly. Is there anything else you want to explore?'

In [28]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search analye the result then perform more searches if needed.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_Cjt0EH8sovZrSu2NLAfhD3zg', name='search', type='function_call', id='fc_04100cfebd6fe50e006a30b845e9d08196af7aa70134ab9541', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': '

In [31]:
!uv add toyaikit

Resolved 127 packages in 4.58s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/902.15 KiB          
⠙ Preparing packages... (0/7)------------------- 14.83 KiB/902.15 KiB        
⠙ Preparing packages... (0/7)------------------- 14.83 KiB/902.15 KiB        
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.83 KiB/902.15 KiB        
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.83 KiB/902.15 KiB        
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.83 KiB/902.15 KiB        
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing pac

In [32]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [33]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [34]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [35]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [36]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [37]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [38]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received
